# GEE CHIRPS Downloader - Komparasi Varian SAT vs RNL (Multi-Band Stacking)
Notebook ini didesain khusus untuk mengunduh dan menyelaraskan dua varian utama **CHIRPS v3.0** dari Google Earth Engine (GEE):
1. **CHIRPS v3 Daily SAT** (`UCSB-CHC/CHIRPS/V3/DAILY_SAT`) - Berbasis murni observasi satelit inframerah CCD.
2. **CHIRPS v3 Daily RNL** (`UCSB-CHC/CHIRPS/V3/DAILY_RNL`) - Berbasis asimilasi latar belakang model reanalisis cuaca atmosfer.

### Fitur Utama:
- **Dual Compatibility:** Dapat berjalan di environment **Lokal** maupun **Kaggle Notebook** secara otomatis.
- **Smart Cache / Input Check:** Mendeteksi dataset yang sudah diunduh sebelumnya di Kaggle Input Dataset.
- **Multi-Band Stacking (`toBands()`):** Mempercepat unduhan hingga 30x lipat tanpa *rate-limit* API.
- **Struktur Folder Terpisah:** `data/chirps_sat/` dan `data/chirps_rnl/`.

In [1]:
!pip install geemap earthengine-api rioxarray geopandas xarray netCDF4 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 45.3 MB/s eta 0:00:00


In [2]:
import ee
import geopandas as gpd
import geemap
from shapely.validation import make_valid
from shapely.ops import transform, unary_union
import os
import shutil
import glob
import rioxarray as rxr
import pandas as pd
import calendar
from datetime import datetime, timedelta

# ==========================================
# 0. PENGATURAN DIREKTORI DINAMIS & PARAMETER
# ==========================================
BASE_DIR = os.getcwd()

if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    file_geojson = "/kaggle/input/datasets/jerismeteo/projek-downscale/33.05_kecamatan.geojson"
    if not os.path.exists(file_geojson):
        found_geo = glob.glob("/kaggle/input/**/33.05_kecamatan.geojson", recursive=True)
        file_geojson = found_geo[0] if found_geo else os.path.join(BASE_DIR, "33.05_kecamatan.geojson")
else:
    file_geojson = os.path.join(BASE_DIR, "33.05_kecamatan.geojson")

FOLDER_OUTPUT_SAT = os.path.join(BASE_DIR, "data", "chirps_sat")
FOLDER_OUTPUT_RNL = os.path.join(BASE_DIR, "data", "chirps_rnl")

# Direktori Dataset CHIRPS yang sudah diunduh sebelumnya di Kaggle
KAGGLE_INPUT_DIR = "/kaggle/input/datasets/jerismeteo/gee-chirps-kebumen"

tahun_awal = 2000
tahun_akhir = 2026

print(f"📌 File GeoJSON        : {file_geojson}")
print(f"📌 Folder Output SAT   : {FOLDER_OUTPUT_SAT}")
print(f"📌 Folder Output RNL   : {FOLDER_OUTPUT_RNL}")
print(f"📌 Periode Pengunduhan : {tahun_awal} s.d. {tahun_akhir}")


📌 File GeoJSON        : /kaggle/input/datasets/jerismeteo/projek-downscale/33.05_kecamatan.geojson
📌 Folder Output SAT   : /kaggle/working/data/chirps_sat
📌 Folder Output RNL   : /kaggle/working/data/chirps_rnl
📌 Periode Pengunduhan : 2000 s.d. 2026


In [3]:
import json
from google.oauth2.service_account import Credentials
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    service_account_info = json.loads(user_secrets.get_secret("GEE_KEY"))
    
    SCOPES = ['https://www.googleapis.com/auth/earthengine']
    credentials = Credentials.from_service_account_info(service_account_info, scopes=SCOPES)
    
    ee.Initialize(credentials=credentials, project='staklimjerukagung')
    print("✅ Berhasil Inisialisasi GEE via Service Account (GEE_KEY)")

except Exception as e:
    print(f"⚠️ Gagal Inisialisasi via Secrets, mencoba auth manual: {e}")
    ee.Authenticate()
    ee.Initialize(project='staklimjerukagung')

✅ Berhasil Inisialisasi GEE via Service Account (GEE_KEY)


In [4]:
# ==========================================
# 2. Persiapan Batas Wilayah (GeoJSON Clean)
# ==========================================
if not os.path.exists(file_geojson):
    raise FileNotFoundError(f"File GeoJSON tidak ditemukan di: {file_geojson}")

gdf = gpd.read_file(file_geojson)
if gdf.crs != "EPSG:4326":
    gdf = gdf.to_crs("EPSG:4326")

print("Membersihkan geometri GeoJSON yang cacat...")
gdf = gdf[gdf.geometry.notna()].copy()

def _to_2d(geom):
    if geom is None or geom.is_empty: return None
    return transform(lambda x, y, z=None: (x, y), geom)

def _extract_polygonal(geom):
    if geom is None or geom.is_empty: return None
    if geom.geom_type in ("Polygon", "MultiPolygon"): return geom
    if geom.geom_type == "GeometryCollection":
        polys = [g for g in geom.geoms if g.geom_type in ("Polygon", "MultiPolygon")]
        if not polys: return None
        return unary_union(polys)
    return None

def _clean_geom(geom):
    if geom is None or geom.is_empty: return None
    geom = _to_2d(geom)
    geom = make_valid(geom)
    geom = _extract_polygonal(geom)
    if geom is None or geom.is_empty: return None
    geom = geom.buffer(0)
    if geom is None or geom.is_empty: return None
    if not geom.is_valid:
        geom = make_valid(geom)
        geom = _extract_polygonal(geom)
    if geom is None or geom.is_empty or not geom.is_valid: return None
    return geom

gdf["geometry"] = gdf["geometry"].apply(_clean_geom)
gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
gdf = gdf[gdf.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
gdf.reset_index(drop=True, inplace=True)

if gdf.empty:
    raise ValueError("Semua geometri tidak valid setelah proses cleaning.")

print("Mengonversi ke Earth Engine...")
geojson_fc = gdf.__geo_interface__
batas_kebumen = ee.FeatureCollection(geojson_fc["features"])

Membersihkan geometri GeoJSON yang cacat...
Mengonversi ke Earth Engine...


In [5]:
# ==========================================
# 3. FUNGSI UNDUH SUPER-CEPAT CHIRPS (SAT & RNL)
# ==========================================
VARIAN_CONFIG = {
    'sat': {
        'collection': 'UCSB-CHC/CHIRPS/V3/DAILY_SAT',
        'label': 'CHIRPS v3 SAT (Satelit)',
        'folder_name': 'chirps_sat'
    },
    'rnl': {
        'collection': 'UCSB-CHC/CHIRPS/V3/DAILY_RNL',
        'label': 'CHIRPS v3 RNL (Reanalisis)',
        'folder_name': 'chirps_rnl'
    }
}

def cari_file_input_chirps(tipe, tahun, bulan):
    """Mengecek apakah file NetCDF CHIRPS sudah tersedia di Input Dataset Kaggle."""
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        possible_paths = [
            f"{KAGGLE_INPUT_DIR}/data/chirps_{tipe}/{tahun}/chirps_{tipe}_{tahun}_{bulan:02d}.nc",
            f"{KAGGLE_INPUT_DIR}/data/chirps/{tahun}/chirps_{tahun}_{bulan:02d}.nc",
            f"{KAGGLE_INPUT_DIR}/{tahun}/chirps_{tipe}_{tahun}_{bulan:02d}.nc",
            f"{KAGGLE_INPUT_DIR}/chirps_{tipe}_{tahun}_{bulan:02d}.nc"
        ]
        for p in possible_paths:
            if os.path.exists(p):
                return p
    return None

def unduh_chirps_varian_bulanan(tipe, tahun, bulan, batas_ee, output_base_dir):
    cfg = VARIAN_CONFIG[tipe]
    out_dir = os.path.join(output_base_dir, str(tahun))
    os.makedirs(out_dir, exist_ok=True)
    
    nc_path = os.path.join(out_dir, f"chirps_{tipe}_{tahun}_{bulan:02d}.nc")
    tif_path = os.path.join(out_dir, f"chirps_{tipe}_{tahun}_{bulan:02d}_temp.tif")
    
    # 1. Cek di folder output kerja
    if os.path.exists(nc_path):
        print(f"[{tipe.upper()} {tahun}-{bulan:02d}] File sudah ada di working directory, dilewati...")
        return
        
    # 2. Cek di Kaggle Input Datasets
    input_file = cari_file_input_chirps(tipe, tahun, bulan)
    if input_file and os.path.exists(input_file):
        print(f"[{tipe.upper()} {tahun}-{bulan:02d}] ✓ Ditemukan di Input Kaggle: {input_file}")
        print(f"[{tipe.upper()} {tahun}-{bulan:02d}] Menyalin file ke {nc_path}...")
        shutil.copy2(input_file, nc_path)
        return
        
    tgl_mulai = f"{tahun}-{bulan:02d}-01"
    if bulan == 12:
        tgl_akhir = f"{tahun+1}-01-01"
    else:
        tgl_akhir = f"{tahun}-{bulan+1:02d}-01"
        
    print(f"[{tipe.upper()} {tahun}-{bulan:02d}] Menarik data ImageCollection dari GEE ({cfg['label']})...")
    try:
        dataset = (ee.ImageCollection(cfg['collection'])
                   .filter(ee.Filter.date(tgl_mulai, tgl_akhir))
                   .select('precipitation'))
        
        count = dataset.size().getInfo()
        if count == 0:
            print(f"[{tipe.upper()} {tahun}-{bulan:02d}] ⚠️ Tidak ada data di GEE untuk periode ini.")
            return
            
        timestamps_ms = dataset.aggregate_array('system:time_start').getInfo()
        time_index = pd.to_datetime(timestamps_ms, unit='ms')
        
        stacked_image = dataset.toBands().clip(batas_ee)
        
        print(f"[{tipe.upper()} {tahun}-{bulan:02d}] Mengunduh {count} hari sekaligus ke TIF sementara...")
        geemap.ee_export_image(
            stacked_image,
            filename=tif_path,
            region=batas_ee.geometry(),
            scale=5566,
            file_per_band=False
        )
        
        with rxr.open_rasterio(tif_path, masked=True) as da:
            da = da.rename({'band': 'time'})
            if len(da.time) == len(time_index):
                da['time'] = time_index
            else:
                da['time'] = pd.date_range(start=time_index[0], periods=len(da.time), freq='D')
                
            da.name = "precipitation"
            da.to_netcdf(nc_path)
            
        if os.path.exists(tif_path): os.remove(tif_path)
        print(f"[{tipe.upper()} {tahun}-{bulan:02d}] ✓ Selesai! Tersimpan di {nc_path}\n")
        
    except Exception as e:
        print(f"[{tipe.upper()} {tahun}-{bulan:02d}] ❌ Error: {e}")
        if os.path.exists(tif_path): os.remove(tif_path)

def unduh_chirps_komparasi_multi_tahun(tahun_awal, tahun_akhir, batas_ee, base_dir):
    print(f"\n{'='*70}")
    print(f"UNDUH CHIRPS v3 KOMPARASI (SAT & RNL): {tahun_awal} - {tahun_akhir}")
    print(f"{'='*70}\n")
    
    folder_sat = os.path.join(base_dir, "data", "chirps_sat")
    folder_rnl = os.path.join(base_dir, "data", "chirps_rnl")
    
    for tahun in range(tahun_awal, tahun_akhir + 1):
        for bulan in range(1, 13):
            print(f"--- Periode {tahun}-{bulan:02d} ---")
            # 1. Unduh CHIRPS SAT
            unduh_chirps_varian_bulanan('sat', tahun, bulan, batas_ee, folder_sat)
            # 2. Unduh CHIRPS RNL
            unduh_chirps_varian_bulanan('rnl', tahun, bulan, batas_ee, folder_rnl)
            
    print(f"{'='*70}")
    print("🎉 SELESAI UNDUH SELURUH VARIAN CHIRPS (SAT & RNL)")
    print(f"{'='*70}")


In [6]:
# ==========================================
# 4. EKSEKUSI PENGUNDUHAN DUA VARIAN CHIRPS
# ==========================================
unduh_chirps_komparasi_multi_tahun(
    tahun_awal=tahun_awal,
    tahun_akhir=tahun_akhir,
    batas_ee=batas_kebumen,
    base_dir=BASE_DIR
)



UNDUH CHIRPS v3 KOMPARASI (SAT & RNL): 2000 - 2026

--- Periode 2000-01 ---
[SAT 2000-01] Menarik data ImageCollection dari GEE (CHIRPS v3 SAT (Satelit))...
[SAT 2000-01] Mengunduh 31 hari sekaligus ke TIF sementara...
Generating URL ...
Please wait ...
Data downloaded to /kaggle/working/data/chirps_sat/2000/chirps_sat_2000_01_temp.tif


Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.


[SAT 2000-01] ✓ Selesai! Tersimpan di /kaggle/working/data/chirps_sat/2000/chirps_sat_2000_01.nc

[RNL 2000-01] Menarik data ImageCollection dari GEE (CHIRPS v3 RNL (Reanalisis))...
[RNL 2000-01] Mengunduh 31 hari sekaligus ke TIF sementara...
Generating URL ...
Please wait ...
Data downloaded to /kaggle/working/data/chirps_rnl/2000/chirps_rnl_2000_01_temp.tif
[RNL 2000-01] ✓ Selesai! Tersimpan di /kaggle/working/data/chirps_rnl/2000/chirps_rnl_2000_01.nc

--- Periode 2000-02 ---
[SAT 2000-02] Menarik data ImageCollection dari GEE (CHIRPS v3 SAT (Satelit))...


Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.


[SAT 2000-02] Mengunduh 29 hari sekaligus ke TIF sementara...
Generating URL ...
Please wait ...
Data downloaded to /kaggle/working/data/chirps_sat/2000/chirps_sat_2000_02_temp.tif
[SAT 2000-02] ✓ Selesai! Tersimpan di /kaggle/working/data/chirps_sat/2000/chirps_sat_2000_02.nc

[RNL 2000-02] Menarik data ImageCollection dari GEE (CHIRPS v3 RNL (Reanalisis))...


...

[SAT 2026-07] Mengunduh 31 hari sekaligus ke TIF sementara...
Generating URL ...
Please wait ...
Data downloaded to /kaggle/working/data/chirps_sat/2026/chirps_sat_2026_07_temp.tif
[SAT 2026-07] ✓ Selesai! Tersimpan di /kaggle/working/data/chirps_sat/2026/chirps_sat_2026_07.nc

[RNL 2026-07] Menarik data ImageCollection dari GEE (CHIRPS v3 RNL (Reanalisis))...


Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.


[RNL 2026-07] Mengunduh 31 hari sekaligus ke TIF sementara...
Generating URL ...
Please wait ...
Data downloaded to /kaggle/working/data/chirps_rnl/2026/chirps_rnl_2026_07_temp.tif
[RNL 2026-07] ✓ Selesai! Tersimpan di /kaggle/working/data/chirps_rnl/2026/chirps_rnl_2026_07.nc

--- Periode 2026-08 ---
[SAT 2026-08] Menarik data ImageCollection dari GEE (CHIRPS v3 SAT (Satelit))...
[SAT 2026-08] ⚠️ Tidak ada data di GEE untuk periode ini.
[RNL 2026-08] Menarik data ImageCollection dari GEE (CHIRPS v3 RNL (Reanalisis))...


Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.


[RNL 2026-08] ⚠️ Tidak ada data di GEE untuk periode ini.
--- Periode 2026-09 ---
[SAT 2026-09] Menarik data ImageCollection dari GEE (CHIRPS v3 SAT (Satelit))...
[SAT 2026-09] ⚠️ Tidak ada data di GEE untuk periode ini.
[RNL 2026-09] Menarik data ImageCollection dari GEE (CHIRPS v3 RNL (Reanalisis))...
[RNL 2026-09] ⚠️ Tidak ada data di GEE untuk periode ini.
--- Periode 2026-10 ---
[SAT 2026-10] Menarik data ImageCollection dari GEE (CHIRPS v3 SAT (Satelit))...
[SAT 2026-10] ⚠️ Tidak ada data di GEE untuk periode ini.
[RNL 2026-10] Menarik data ImageCollection dari GEE (CHIRPS v3 RNL (Reanalisis))...
[RNL 2026-10] ⚠️ Tidak ada data di GEE untuk periode ini.
--- Periode 2026-11 ---
[SAT 2026-11] Menarik data ImageCollection dari GEE (CHIRPS v3 SAT (Satelit))...
[SAT 2026-11] ⚠️ Tidak ada data di GEE untuk periode ini.
[RNL 2026-11] Menarik data ImageCollection dari GEE (CHIRPS v3 RNL (Reanalisis))...
[RNL 2026-11] ⚠️ Tidak ada data di GEE untuk periode ini.
--- Periode 2026-12 ---
[SA